In [2]:
# =============================================================================
# DAY 4.0 — EXPORT DASHBOARD SOURCES  (HARD GUARD: needs Day-2 outputs)
# =============================================================================
from pathlib import Path
import pandas as pd, numpy as np

FACT_SRC = Path("covid_derived.parquet")  # Day-2 calendar-safe fact
CTRY_SRC = Path("country_rollout_vs_death.csv")  # Day-2 per-country scalars
if not (FACT_SRC.exists() and CTRY_SRC.exists()):
    raise FileNotFoundError(
        "Day 4 needs Day-2 outputs (covid_derived.parquet + country_rollout_vs_death.csv). "
        "Run the Day-2 derivation cell FIRST — building on the gappy covid_subset would "
        "ship a row-aligned (not day-aligned) 21-day lag. Do not recompute the lag here."
    )

fact = pd.read_parquet(FACT_SRC)
cty = pd.read_csv(CTRY_SRC, index_col=0)  # was ctry  ->  cty
fact["date"] = pd.to_datetime(fact["date"])

# sanity: the frame feeding the dashboard MUST be a rectangle (calendar-safe)
sizes = fact.groupby("iso_code").size()
assert (
    sizes.nunique() == 1
), "derived frame is NOT calendar-continuous -> Day-2 reindex failed"
assert not fact.duplicated(
    ["iso_code", "date"]
).any(), "fact not unique per (country,date)"
print(
    f"[export] fact {fact.shape} (rect: {sizes.iloc[0]} days x {fact.iso_code.nunique()} countries) "
    f"| country dim {ctry.shape}"
)

# ---- fact for the time-series pages (keep statics + the rolling/derived cols) ----
KEEP = [
    "iso_code",
    "location",
    "continent",
    "date",
    "new_cases_per_million_7d",
    "new_deaths_per_million_7d",
    "people_fully_vaccinated_per_hundred_7d",
    "cfr_lagged_28d",
    "cfr_naive_28d",
    "doubling_time_14d",
    "is_wave_peak",
]
KEEP = [c for c in KEEP if c in fact.columns]
dash_fact = fact[KEEP].sort_values(["iso_code", "date"]).reset_index(drop=True)

# ---- country dim for the scatter (one row per country; the scatter's ONLY source) ----
dash_country = cty.reset_index().rename(columns={"index": "iso_code"})

# ---- tidy long for a Power BI "metric" slicer on the line chart ----
MELT = [
    c
    for c in [
        "new_cases_per_million_7d",
        "new_deaths_per_million_7d",
        "people_fully_vaccinated_per_hundred_7d",
        "cfr_lagged_28d",
        "doubling_time_14d",
    ]
    if c in dash_fact.columns
]
dash_tidy = (
    dash_fact.melt(
        id_vars=["iso_code", "location", "continent", "date"],
        value_vars=MELT,
        var_name="metric",
        value_name="value",
    )
    .dropna(subset=["value"])
    .sort_values(["iso_code", "date", "metric"])
)

dash_fact.to_csv("dash_fact.csv", index=False)
dash_country.to_csv("dash_country.csv", index=False)
dash_tidy.to_csv("dash_tidy_long.csv", index=False)
print(
    f"[export] wrote dash_fact.csv {dash_fact.shape} | dash_country.csv {dash_country.shape} "
    f"| dash_tidy_long.csv {dash_tidy.shape}"
)
print("Columns present in fact:", [c for c in KEEP])

[export] fact (11676, 82) (rect: 1668 days x 7 countries) | country dim (7, 9)
[export] wrote dash_fact.csv (11676, 11) | dash_country.csv (7, 10) | dash_tidy_long.csv (41998, 6)
Columns present in fact: ['iso_code', 'location', 'continent', 'date', 'new_cases_per_million_7d', 'new_deaths_per_million_7d', 'people_fully_vaccinated_per_hundred_7d', 'cfr_lagged_28d', 'cfr_naive_28d', 'doubling_time_14d', 'is_wave_peak']
